**Install modules**

In [ ]:
%%capture
%pip install pandas
%pip install openpyxl

In [6]:
import pandas as pd
import openpyxl

In [18]:
df = pd.read_csv(f"./data/extracted/csv/continuous.csv", encoding = "utf-8")
df = df.fillna("")
globals()[f"continuous"] = df
df.head()

,Unnamed: 0.1,Unnamed: 0,id,study,subgroup,outcome,data_type,ni,yi,sdi,...,q1,median,q3,max,age_mean,age_sd,male,female,follow-up_mo,follow-up_yr
0,0,0,10,"Barié et al., 2020",BPTB,ikdc_subjective,continuous,43.0,91.0,7.3,...,,,,,30.6,7.5,17.0,13.0,122.4,10.2
1,1,1,10,"Barié et al., 2020",QT,ikdc_subjective,continuous,43.0,92.0,11.5,...,,,,,30.6,7.5,17.0,13.0,122.4,10.2
2,2,2,12,"Bi et al., 2018",HT,ikdc_subjective,continuous,62.0,90.4,7.1,...,,,,,27.9,6.7,31.0,31.0,30.3,2.5
3,3,3,12,"Bi et al., 2018",PLT,ikdc_subjective,continuous,62.0,89.3,8.4,...,,,,,27.9,6.7,31.0,31.0,30.3,2.5
4,4,4,13,"Björnsson et al., 2016",BPTB,ikdc_subjective,continuous,61.0,67.3,20.8,...,,,,,28.2,9.1,42.0,19.0,202.6,16.9


In [15]:
df.to_csv("./data/extracted/csv/continuous.csv", encoding = "utf-8")

**Import extracted data** (continuous outcomes)\
`./data/extracted`

In [8]:
outcomes = [
    "ikdc_subjective",
    "lysholm",
    "tegner",
    "instrumented_laxity"
]

for x in outcomes:
    df = continuous[continuous["outcome"] == f"{x}"]
    df = df.drop
    df.to_csv(f"./data/extracted/csv/{x}.csv", encoding = "utf-8")
    df.to_excel(f"./data/extracted/xlsx/{x}.xlsx")
    globals()[x] = df

2. **Pivot the data** (one row per study)\
`./data/reformatted`

In [24]:
dfs = {
    "ikdc_subjective": ikdc_subjective, 
    "lysholm": lysholm, 
    "tegner": tegner, 
    "instrumented_laxity": instrumented_laxity
}

for name, df in dfs.items():
    df = pd.read_csv(f"./data/extracted/csv/{x}.csv", encoding = "utf-8")

    df["group"] = df.groupby(["study"]).cumcount() + 1
    
    # reshape
    wide = (
        df.pivot(
            index= "id",
            columns="group",
            values=["study", "subgroup", "ni", "yi", "sdi", 
                    "age_mean", "age_sd", 
                    "male", "female", 
                    "follow-up_mo", "follow-up_yr"]
        )
    )
    
    # flatten column names
    wide.columns = [
        f"{name}{group}" if name != "subgroup" else f"treat{group}"
        for name, group in wide.columns
    ]
    
    # rename variables
    wide = (
        wide.rename(columns={
            "ni1": "n1",
            "yi1": "m1",
            "sdi1": "sd1",
            "ni2": "n2",
            "yi2": "m2",
            "sdi2": "sd2"
            
        })
        .reset_index()
    )

    wide["n1"] = wide["n1"].astype(int)
    wide["n2"] = wide["n2"].astype(int)
    wide["sd1"] = wide["sd1"].astype(float)
    wide["sd2"] =wide["sd2"].astype(float)
    wide["m1"] = wide["m1"].astype(float)
    wide["m2"] = wide["m2"].astype(float)
    
    import os
    os.makedirs(f"./data/reformatted/csv", exist_ok = True)
    wide.to_csv(f"./data/reformatted/csv/{name}.csv", encoding = "utf-8")
    
    os.makedirs(f"./data/reformatted/xlsx", exist_ok = True)
    wide.to_excel(f"./data/reformatted/xlsx/{name}.xlsx")
    
    globals()[f"{name}_wide"] = wide

In [25]:
ikdc_subjective_wide.head()

,id,study1,study2,treat1,treat2,n1,n2,m1,m2,sd1,...,age_sd1,age_sd2,male1,male2,female1,female2,follow-up_mo1,follow-up_mo2,follow-up_yr1,follow-up_yr2
0,2,"Aglietti et al., 2004","Aglietti et al., 2004",BPTB,HT,60,60,2.20,1.95,1.08,...,4.97,4.97,46.0,46.0,14.0,14.0,24.0,24.0,2.0,2.0
1,3,"Ahldén et al., 2009","Ahldén et al., 2009",BPTB,HT,25,22,2.60,1.40,3.30,...,8.9,8.9,14.0,14.0,8.0,8.0,90.271348,90.271348,7.5,7.5
2,5,"Anderson et al., 2001","Anderson et al., 2001",BPTB,HT,35,33,2.10,3.10,2.00,...,7.13,7.13,23.0,23.0,12.0,12.0,34.6,34.6,2.9,2.9
3,7,"Aune et al., 2001","Aune et al., 2001",BPTB,HT,32,29,3.30,2.90,2.60,...,7.0,7.0,19.0,19.0,16.0,16.0,24.0,24.0,2.0,2.0
4,10,"Barié et al., 2020","Barié et al., 2020",BPTB,QT,43,43,1.05,1.00,1.36,...,7.5,7.5,17.0,17.0,13.0,13.0,122.4,122.4,10.2,10.2


3. **Prepare sheets for analysis** (calculate standardized mean differences and standard errors of standardized mean differences as TE and seTE, respectively)\
`./data/sheets_for_analysis`

In [28]:
import numpy as np

dfs = {
    "ikdc_subjective": ikdc_subjective, 
    "lysholm": lysholm, 
    "tegner": tegner, 
    "instrumented_laxity": instrumented_laxity
}

for name, df in dfs.items():
    # Standardized Mean Difference (SMD)
    df = pd.read_csv(f"./data/reformatted/csv/{name}.csv", encoding = "utf-8")

    def SMD(n1, n2, m1, m2, sd1, sd2):
        sd = (
            (((n1 - 1) * sd1**2) + ((n2 - 1) * sd2**2))
            / (n1 + n2 - 2)
        )
        sp = np.sqrt(sd)
        return round((np.abs(m1 - m2) / sp), 3)
    
    # Standard Error of the Standardized Mean Difference (SMD)
    def seSMD(n1, n2, smd):
        se = np.sqrt(
            (n1 + n2)/(n1*n2)
            + smd**2/(2*(n1+n2-2))
        )
        se = round(se, 3)
        return se

    df["TE"] = SMD(df["n1"], df["n2"], df["y1"], df["y2"], df["sd1"], df["sd2"])
    df["seTE"] = seSMD(df["n1"], df["n2"], df["TE"])

    df.to_csv(f"./data/sheets_for_analysis/csv/{name}.csv", encoding = "utf-8")
    df.to_excel(f"./data/sheets_for_analysis/xlsx/{name}.xlsx")

    globals()[name] = df

**Display preview dataframes** (to check for correct columns for network meta-analysis)  

In [19]:
ikdc_subjective.head()

,id,study1,study2,treat1,treat2,n1,n2,m1,m2,sd1,...,age_mean1,age_mean2,male1,male2,female1,female2,follow-up_mo1,follow-up_mo2,TE,seTE
0,2,"Aglietti et al., 2004","Aglietti et al., 2004",BPTB,HT,60,60,2.20,1.95,1.08,...,25.0,25.0,46.0,46.0,14.0,14.0,24.0,24.0,0.209,0.183
1,3,"Ahldén et al., 2009","Ahldén et al., 2009",BPTB,HT,25,22,2.60,1.40,3.30,...,27.41,27.41,14.0,14.0,8.0,8.0,90.271348,90.271348,0.401,0.295
2,5,"Anderson et al., 2001","Anderson et al., 2001",BPTB,HT,35,33,2.10,3.10,2.00,...,23.6,23.6,23.0,23.0,12.0,12.0,34.6,34.6,0.465,0.246
3,7,"Aune et al., 2001","Aune et al., 2001",BPTB,HT,32,29,3.30,2.90,2.60,...,25.0,25.0,19.0,19.0,16.0,16.0,24.0,24.0,0.160,0.257
4,10,"Barié et al., 2020","Barié et al., 2020",BPTB,QT,43,43,1.05,1.00,1.36,...,30.6,30.6,17.0,17.0,13.0,13.0,122.4,122.4,0.040,0.216


In [20]:
lysholm.head()

,id,study1,study2,treat1,treat2,n1,n2,m1,m2,sd1,...,age_mean1,age_mean2,male1,male2,female1,female2,follow-up_mo1,follow-up_mo2,TE,seTE
0,2,"Aglietti et al., 2004","Aglietti et al., 2004",BPTB,HT,60,60,2.20,1.95,1.08,...,25.0,25.0,46.0,46.0,14.0,14.0,24.0,24.0,0.209,0.183
1,3,"Ahldén et al., 2009","Ahldén et al., 2009",BPTB,HT,25,22,2.60,1.40,3.30,...,27.41,27.41,14.0,14.0,8.0,8.0,90.271348,90.271348,0.401,0.295
2,5,"Anderson et al., 2001","Anderson et al., 2001",BPTB,HT,35,33,2.10,3.10,2.00,...,23.6,23.6,23.0,23.0,12.0,12.0,34.6,34.6,0.465,0.246
3,7,"Aune et al., 2001","Aune et al., 2001",BPTB,HT,32,29,3.30,2.90,2.60,...,25.0,25.0,19.0,19.0,16.0,16.0,24.0,24.0,0.160,0.257
4,10,"Barié et al., 2020","Barié et al., 2020",BPTB,QT,43,43,1.05,1.00,1.36,...,30.6,30.6,17.0,17.0,13.0,13.0,122.4,122.4,0.040,0.216


In [21]:
tegner.head()

,id,study1,study2,treat1,treat2,n1,n2,m1,m2,sd1,...,age_mean1,age_mean2,male1,male2,female1,female2,follow-up_mo1,follow-up_mo2,TE,seTE
0,2,"Aglietti et al., 2004","Aglietti et al., 2004",BPTB,HT,60,60,2.20,1.95,1.08,...,25.0,25.0,46.0,46.0,14.0,14.0,24.0,24.0,0.209,0.183
1,3,"Ahldén et al., 2009","Ahldén et al., 2009",BPTB,HT,25,22,2.60,1.40,3.30,...,27.41,27.41,14.0,14.0,8.0,8.0,90.271348,90.271348,0.401,0.295
2,5,"Anderson et al., 2001","Anderson et al., 2001",BPTB,HT,35,33,2.10,3.10,2.00,...,23.6,23.6,23.0,23.0,12.0,12.0,34.6,34.6,0.465,0.246
3,7,"Aune et al., 2001","Aune et al., 2001",BPTB,HT,32,29,3.30,2.90,2.60,...,25.0,25.0,19.0,19.0,16.0,16.0,24.0,24.0,0.160,0.257
4,10,"Barié et al., 2020","Barié et al., 2020",BPTB,QT,43,43,1.05,1.00,1.36,...,30.6,30.6,17.0,17.0,13.0,13.0,122.4,122.4,0.040,0.216


In [22]:
instrumented_laxity.head()

,id,study1,study2,treat1,treat2,n1,n2,m1,m2,sd1,...,age_mean1,age_mean2,male1,male2,female1,female2,follow-up_mo1,follow-up_mo2,TE,seTE
0,2,"Aglietti et al., 2004","Aglietti et al., 2004",BPTB,HT,60,60,2.20,1.95,1.08,...,25.0,25.0,46.0,46.0,14.0,14.0,24.0,24.0,0.209,0.183
1,3,"Ahldén et al., 2009","Ahldén et al., 2009",BPTB,HT,25,22,2.60,1.40,3.30,...,27.41,27.41,14.0,14.0,8.0,8.0,90.271348,90.271348,0.401,0.295
2,5,"Anderson et al., 2001","Anderson et al., 2001",BPTB,HT,35,33,2.10,3.10,2.00,...,23.6,23.6,23.0,23.0,12.0,12.0,34.6,34.6,0.465,0.246
3,7,"Aune et al., 2001","Aune et al., 2001",BPTB,HT,32,29,3.30,2.90,2.60,...,25.0,25.0,19.0,19.0,16.0,16.0,24.0,24.0,0.160,0.257
4,10,"Barié et al., 2020","Barié et al., 2020",BPTB,QT,43,43,1.05,1.00,1.36,...,30.6,30.6,17.0,17.0,13.0,13.0,122.4,122.4,0.040,0.216
